# Chatbot Development

Use this notebook to load the model and then initialize, update, and test the chatbot.

### Setup and Imports

In [1]:
import torch
from huggingface_hub import login


from src.chat import Chatbot
from config import BASE_MODEL, MY_MODEL

In [2]:
"""
TODO: Add your Hugging Face token
Options:
1. Use login() and enter token when prompted. It won't ask for your token if you already logged in using the command: huggingface-cli login in the terminal.
2. Set environment variable HUGGINGFACE_TOKEN
3. Pass token directly (not recommended for shared notebooks)
"""

login()



### Initialize and test chatbot

In [3]:
"""
Create chatbot instance using chat.py
"""
chatbot = Chatbot()

In [4]:
"""
Test out generating some responses from the chatbot.
Inference time
"""
test_question = "I am struggling with my classes and have a lot of stress. What options are available for someone in my situation?"

print(f"\nQuestion: {test_question}")
response = chatbot.get_response(test_question)
print(f"Response: {response}")



Question: I am struggling with my classes and have a lot of stress. What options are available for someone in my situation?
Response: It sounds like you're feeling overwhelmed with your classes and experiencing stress. There are various options that might be helpful for you. Before we explore those, I'd like to clarify a few things to better understand your situation:

1. Do you think your stress and academic struggles are related to substance use or addiction?
2. Have you experienced any mental health issues such as anxiety, depression, or burnout?
3. Are you open to seeking professional help, or would you prefer online resources and support groups?
4. Do you have a preferred location for treatment, or are you open to considering options across the country?
5. What is your budget like for treatment? Do you have private insurance, or would you prefer free or low-cost options?
6. Are there any specific therapies or approaches you're interested in, such as cognitive-behavioral therapy (

# TODO: Update pre-trained Llama to be a task-specific chatbot

This part is up to you! You might want to finetune the model, simply make a really good system prompt, use RAG, provide the model relevant data in-context, etc. Be creative!

You can also feel free to do this in another script and then evaluate the model here.

Tips:
- HuggingFace has built-in methods to finetune models, if you choose that route. Take advantage of those methods! You can then save your new, finetuned model in the HuggingFace Hub. Change MY_MODEL in config.py to the name of the model in the hub to make your chatbot use it.
- You may also want to consider LoRA if you choose finetuning.

In [ ]:
# 评估聊天机器人性能

import nltk
from nltk.translate.bleu_score import sentence_bleu
from sklearn.metrics import precision_score, recall_score
import pandas as pd

# 下载nltk数据（如果需要）
# nltk.download('punkt')

# 定义测试数据集
test_cases = [
    {
        "input": "I need help with opioid addiction in Boston.",
        "expected_response_keywords": ["treatment", "facility", "help"],
        "expected_facilities": ["facility_name1", "facility_name2"]  # 根据实际数据调整
    },
    {
        "input": "What mental health services are available?",
        "expected_response_keywords": ["mental health", "options", "location"],
        "expected_facilities": []
    }
]

def evaluate_response_bleu(generated, expected):
    """使用BLEU分数评估响应相似性"""
    generated_tokens = nltk.word_tokenize(generated.lower())
    expected_tokens = [nltk.word_tokenize(expected.lower())]
    return sentence_bleu(expected_tokens, generated_tokens)

def evaluate_keywords(response, keywords):
    """检查响应是否包含预期关键词"""
    response_lower = response.lower()
    return sum(1 for keyword in keywords if keyword in response_lower) / len(keywords)

def evaluate_retrieval(retrieved_facilities, expected_facilities):
    """评估检索准确性"""
    if not expected_facilities:
        return 1.0 if not retrieved_facilities else 0.0
    
    retrieved_names = [f.get('facility_name', '') for f in retrieved_facilities]
    precision = len(set(retrieved_names) & set(expected_facilities)) / len(retrieved_names) if retrieved_names else 0
    recall = len(set(retrieved_names) & set(expected_facilities)) / len(expected_facilities) if expected_facilities else 0
    return {"precision": precision, "recall": recall}

# 运行评估
results = []
for i, case in enumerate(test_cases):
    print(f"测试案例 {i+1}: {case['input']}")
    
    # 重置聊天机器人
    chatbot.clear_memory()
    
    # 生成响应
    response = chatbot.get_response(case['input'])
    
    # 评估响应质量
    bleu_score = evaluate_response_bleu(response, " ".join(case['expected_response_keywords']))  # 简化的BLEU
    keyword_score = evaluate_keywords(response, case['expected_response_keywords'])
    
    # 评估检索（如果有设施返回）
    # 注意：需要从响应中提取设施信息，这里简化假设
    retrieved_facilities = []  # 从响应或内存中提取
    retrieval_scores = evaluate_retrieval(retrieved_facilities, case['expected_facilities'])
    
    results.append({
        "case": i+1,
        "response": response,
        "bleu_score": bleu_score,
        "keyword_score": keyword_score,
        "retrieval_precision": retrieval_scores["precision"],
        "retrieval_recall": retrieval_scores["recall"]
    })
    
    print(f"响应: {response}")
    print(f"BLEU分数: {bleu_score:.3f}, 关键词覆盖: {keyword_score:.3f}")
    print(f"检索精确度: {retrieval_scores['precision']:.3f}, 召回率: {retrieval_scores['recall']:.3f}")
    print("-" * 50)

# 计算平均分数
avg_bleu = sum(r['bleu_score'] for r in results) / len(results)
avg_keyword = sum(r['keyword_score'] for r in results) / len(results)
avg_precision = sum(r['retrieval_precision'] for r in results) / len(results)
avg_recall = sum(r['retrieval_recall'] for r in results) / len(results)

print("
总体评估结果:")
print(f"平均BLEU分数: {avg_bleu:.3f}")
print(f"平均关键词覆盖: {avg_keyword:.3f}")
print(f"平均检索精确度: {avg_precision:.3f}")
print(f"平均检索召回率: {avg_recall:.3f}")